# Create Blavatnik Awards (PRIZE PATTERN)

Blavatnik Awards for Young Scientists honorees, scraped from the awarding body's
own static-HTML profile pages on `blavatnikawards.org`. Source authority is the
awarding body (the Foundation contracts the New York Academy of Sciences to
administer the Awards and host the public site). No Wikipedia / Wikidata backfill.

**Prerequisite:** run `scripts/local/blavatnik_to_s3.py` to scrape the 4 regional
listing pages plus per-honoree profile pages and upload parquet to S3.

**Source:** `https://blavatnikawards.org/honorees/` plus regional listings
(`national-finalists`, `uk-honorees`, `israel-laureates`) and per-honoree pages
at `/honorees/profile/{slug}/`.

**S3 location:** `s3://openalex-ingest/awards/blavatnik/blavatnik_awards.parquet`

**Awarding body:** Blavatnik Family Foundation — OpenAlex `F4320312914`,
DOI `10.13039/100011643`. ROR returns NULL from the public OpenAlex API at Step 0.

## Prize-pattern modeling

- **One row per (year × region × honoree).** The same person can be honored in
  multiple years (e.g. Regional Finalist → National Laureate) — each appears as
  a distinct row with its own profile URL.
- **`lead_investigator`** is the honoree (given_name, family_name, current
  position, institution).
- **`funding_type`** is `prize`.
- **`funder_scheme`** is the regional × role × status descriptor
  (e.g. `Blavatnik National Award - Faculty Laureate`).
- **Amount logic.** Per the official Blavatnik About page, each year three
  Finalists are selected per category and one becomes the Laureate, winning
  **US$250,000** in unrestricted funds. The non-Laureate Finalists' amounts
  are not publicly priced on the awarding body's site. So:
  - Laureate → `amount = 250000`, `currency = USD`
  - Finalist → `amount` NULL, `currency` NULL (explicit Step 6.7 waiver)
- **`funder_award_id`** is a synthetic key combining year + region + role +
  profile slug; the source script raises on collisions.
- **`declined` boolean** column emitted always-False for schema parity with
  Fields Medal / Abel Prize. No Blavatnik honoree has declined per the awarding
  body's site.

## Expected raw fields (from script)

- `profile_slug`, `profile_url` — stable per-honoree identifiers
- `laureate_name`, `laureate_given_name`, `laureate_family_name`
- `award_year` (int), `region`, `status` (Laureate/Finalist), `role` (Faculty/Postdoctoral)
- `h2_raw` — source h2 text, preserved verbatim for provenance
- `current_position`, `institution`, `discipline`
- `citation` ("Recognized for:" text), `research_summary`
- `amount_usd` (int or NULL), `currency` (USD or NULL)
- `declined` (always False)
- `funder_award_id` (synthetic, dedupe-checked in script)
- `downloaded_at`


## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.blavatnik_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/blavatnik/blavatnik_awards.parquet`;

## Step 1.5: Inspect raw data before transform

Per runbook §1.5 (lines 377-440): inspect columns, sample rows, scan for money /
currency fields before writing transformation SQL. The Blavatnik scrape uses
`amount_usd` as the canonical amount column (apportionment is per-row at
script-write time, not per-CTE here) and hardcodes `currency = USD` for
Laureates; the scan is a sanity check that no other money-shaped column slipped
in via raw scrape.

In [ ]:
%sql
SELECT COUNT(*) AS raw_rows FROM openalex.awards.blavatnik_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.blavatnik_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.blavatnik_raw LIMIT 10;

In [ ]:
%sql
-- Required money-field scan per runbook §1.5. Expected hits: amount_usd.
-- Any other column name matching the money regex would mean the scrape
-- inadvertently surfaced an unmapped amount field.
SELECT column_name
FROM (DESCRIBE openalex.awards.blavatnik_raw)
WHERE LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|fund_|funding|cost|budget|grant_offer|awarded';

In [ ]:
%sql
-- Required currency-field scan per runbook §1.5. Expected hit: currency.
SELECT column_name
FROM (DESCRIBE openalex.awards.blavatnik_raw)
WHERE LOWER(column_name) RLIKE 'currenc|ccy|iso_4217';

In [ ]:
%sql
-- Sanity-check the amount distribution (Laureates only have amount; Finalists are NULL).
-- Real grant-amount distribution should sit between $250,000 (single Laureate)
-- and ~$42M cumulative across all Laureates.
SELECT
    MIN(TRY_CAST(amount_usd AS DOUBLE)) AS min_amt,
    MAX(TRY_CAST(amount_usd AS DOUBLE)) AS max_amt,
    AVG(TRY_CAST(amount_usd AS DOUBLE)) AS avg_amt,
    SUM(TRY_CAST(amount_usd AS DOUBLE)) AS sum_amt,
    COUNT(amount_usd) AS rows_with_amount,
    COUNT(*) AS total_rows
FROM openalex.awards.blavatnik_raw;

## Step 1.6: Funder existence fail-fast

Runbook §2.2.4 (lines 442-458) — mandatory for every ingest. Must return exactly
1 row. If 0 rows, STOP — the funder is missing from `openalex.common.funder` and
the Step 2 `CROSS JOIN` would silently emit zero rows.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320312914;

## Step 2: Transform to OpenAlex awards schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.blavatnik_awards
USING delta
AS
WITH blavatnik_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320312914
), normalized AS (
    SELECT
        *,
        TRY_CAST(award_year AS INT) AS award_year_int,
        TRY_CAST(amount_usd AS DOUBLE) AS amount_double,
        NULLIF(TRIM(currency), '') AS currency_norm,
        NULLIF(TRIM(citation), '') AS citation_norm,
        NULLIF(TRIM(institution), '') AS institution_norm,
        NULLIF(TRIM(current_position), '') AS current_position_norm,
        NULLIF(TRIM(laureate_given_name), '') AS given_name_norm,
        NULLIF(TRIM(laureate_family_name), '') AS family_name_norm,
        NULLIF(TRIM(laureate_name), '') AS laureate_name_norm,
        NULLIF(TRIM(region), '') AS region_norm,
        NULLIF(TRIM(role), '') AS role_norm,
        NULLIF(TRIM(status), '') AS status_norm
    FROM openalex.awards.blavatnik_raw
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':blavatnik:', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    CONCAT(
        CAST(n.award_year_int AS STRING), ' Blavatnik ',
        COALESCE(n.region_norm, 'Unknown'), ' Award ',
        COALESCE(n.status_norm, 'Honoree'),
        CASE WHEN n.role_norm IS NOT NULL THEN CONCAT(' - ', n.role_norm) ELSE '' END,
        ' - ', n.laureate_name_norm
    ) AS display_name,
    CASE
        WHEN TRY_CAST(n.declined AS BOOLEAN) AND n.citation_norm IS NOT NULL
            THEN CONCAT('Declined the prize. ', n.citation_norm)
        WHEN TRY_CAST(n.declined AS BOOLEAN) THEN 'Declined the prize.'
        ELSE n.citation_norm
    END AS description,
    f.funder_id,
    n.funder_award_id,
    n.amount_double AS amount,
    n.currency_norm AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name, f.ror_id, f.doi
    ) AS funder,
    'prize' AS funding_type,
    CONCAT(
        'Blavatnik ', COALESCE(n.region_norm, 'Unknown'), ' Award - ',
        COALESCE(n.role_norm, 'Honoree'), ' ',
        COALESCE(n.status_norm, 'Honoree')
    ) AS funder_scheme,
    'blavatnikawards_org' AS provenance,
    -- Blavatnik ceremony is annually in September per the awarding body's
    -- About page; use September 1 of the award year as canonical date.
    TRY_TO_DATE(CONCAT(CAST(n.award_year_int AS STRING), '-09-01'), 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(CONCAT(CAST(n.award_year_int AS STRING), '-09-30'), 'yyyy-MM-dd') AS end_date,
    n.award_year_int AS start_year,
    n.award_year_int AS end_year,
    struct(
        n.given_name_norm AS given_name,
        n.family_name_norm AS family_name,
        CAST(NULL AS STRING) AS orcid,
        CAST(NULL AS DATE) AS role_start,
        struct(
            n.institution_norm AS name,
            CAST(NULL AS STRING) AS country,
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
        ) AS affiliation
    ) AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.profile_url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':blavatnik:', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM normalized n
CROSS JOIN blavatnik_funder f
WHERE n.funder_award_id IS NOT NULL
  AND n.award_year_int IS NOT NULL
  AND n.laureate_name_norm IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 64

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'blavatnikawards_org' AND priority = 64;

-- Insert into openalex_awards_raw with priority
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    64 AS priority  -- Blavatnik priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.blavatnik_awards;

## Verification

In [ ]:
%sql
-- §6.1 Basic count
SELECT COUNT(*) AS total FROM openalex.awards.blavatnik_awards;

In [ ]:
%sql
-- §6.3 Data completeness (runbook §6.3 form).
-- has_amount expected ~46% (Laureates only — Finalists waived per Step 6.7 note).
-- has_pi expected ~100% (every honoree has a lead_investigator struct built).
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_pi,
    COUNT(funder_award_id) AS has_funder_award_id,
    COUNT(landing_page_url) AS has_landing_page_url,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) AS pct_title,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) AS pct_dates
FROM openalex.awards.blavatnik_awards;

In [ ]:
%sql
-- §6.7 amount/currency fail-fast check.
-- Prize-pattern partial waiver: amount is set only for Laureates ($250,000
-- each per the awarding body's About page); Finalists are documented as
-- not publicly priced. Expected pct_amount ~ Laureates/(Laureates+Finalists)
-- (around 46% as of 2026). currency = {USD} for amount-bearing rows.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    AVG(amount) AS avg_amount,
    SUM(amount) AS total_amount
FROM openalex.awards.blavatnik_awards;

In [ ]:
%sql
-- Duplicate OpenAlex award id check (should be zero rows). The source script
-- already raises on duplicate funder_award_id collisions, but this verifies
-- the post-hash id is also unique.
SELECT id, COUNT(*) AS dupes
FROM openalex.awards.blavatnik_awards
GROUP BY id
HAVING COUNT(*) > 1;

In [ ]:
%sql
-- Funder distribution (should be exactly Blavatnik Family Foundation).
SELECT funder_id, funder.display_name, COUNT(*) AS rows
FROM openalex.awards.blavatnik_awards
GROUP BY funder_id, funder.display_name;

In [ ]:
%sql
-- Year distribution. Expected coverage 2007-2026 (~20 years).
SELECT start_year, COUNT(*) AS rows
FROM openalex.awards.blavatnik_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year DESC;

In [ ]:
%sql
-- funder_scheme distribution. Should reflect 4 regions × 2 roles × 2 statuses.
SELECT funder_scheme, COUNT(*) AS rows
FROM openalex.awards.blavatnik_awards
GROUP BY funder_scheme
ORDER BY rows DESC;

In [ ]:
%sql
-- Sample inspection (sorted by year descending).
SELECT
    SUBSTRING(display_name, 1, 100) AS display_name,
    funder_scheme,
    start_year,
    amount,
    currency,
    lead_investigator.given_name,
    lead_investigator.family_name,
    lead_investigator.affiliation.name AS institution,
    landing_page_url
FROM openalex.awards.blavatnik_awards
ORDER BY start_year DESC, funder_scheme, lead_investigator.family_name
LIMIT 20;

In [ ]:
%sql
-- Confirm insertion into openalex_awards_raw at priority 64.
SELECT provenance, priority, COUNT(*) AS inserted_rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'blavatnikawards_org' AND priority = 64
GROUP BY provenance, priority;